In [0]:
%sql
CREATE TABLE IF NOT EXISTS globalretail.silver.dim_date (
    date_key INT,
    calendar_date DATE,
    day_of_month INT,
    month_num INT,
    month_name STRING,
    quarter_num INT,
    year_num INT,
    week_of_year INT,
    day_of_week_num INT,
    day_name STRING,
    is_weekend BOOLEAN
) USING DELTA;

In [0]:
from datetime import date, timedelta

rows = []
current = date(2025, 1, 1)
end_date = date(2027, 12, 31)
while current <= end_date:
    rows.append((
        int(current.strftime("%Y%m%d")),
        current.isoformat(),
        current.day,
        current.month,
        current.strftime("%B"),
        (current.month - 1) // 3 + 1,
        current.year,
        int(current.strftime("%U")),
        current.isoweekday(),
        current.strftime("%A"),
        current.weekday() >= 5,
    ))
    current += timedelta(days=1)

df = spark.createDataFrame(rows, schema="""
    date_key INT,
    calendar_date STRING,
    day_of_month INT,
    month_num INT,
    month_name STRING,
    quarter_num INT,
    year_num INT,
    week_of_year INT,
    day_of_week_num INT,
    day_name STRING,
    is_weekend BOOLEAN
""").selectExpr(
    "date_key",
    "cast(calendar_date as date) as calendar_date",
    "day_of_month",
    "month_num",
    "month_name",
    "quarter_num",
    "year_num",
    "week_of_year",
    "day_of_week_num",
    "day_name",
    "is_weekend"
)

df.createOrReplaceTempView("vw_dim_date")
spark.sql(f"""
MERGE INTO globalretail.silver.dim_date t
USING vw_dim_date s
ON t.date_key = s.date_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")